In [ ]:

# from .nn_wrappers import SklearnPyTorchNN, SklearnLassoNet
import sys
# sys.path not needed after pip install qtml
from autoarima import autoarima
from autoarimax import autoarimax
from SF_AutoARIMA import SF_AutoARIMA
from SF_AutoARIMAX import SF_AutoARIMAX
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge





models = {
    "AutoARIMA": (autoarima(seasonal=True), {}, 1),
    #"AutoARIMAX": (autoarimax(seasonal=True), {}, 1),
    "SF_AutoARIMA": (SF_AutoARIMA(season_length=1), {}, 1),
    # "SF_AutoARIMAX": (SF_AutoARIMAX(season_length=1), {}, 1),







    "XGBoost":   (XGBRegressor(), {"n_estimators": [100, 200], "max_depth": [3, 5, 7]}, -1),

    # "Ridge": (Ridge(), {"alpha": [0.01, 0.1, 1.0, 10.0]}, -1),
    "LightGBM": (
        LGBMRegressor(random_state=42, verbose=-1),
        {"max_depth": [3, 5], "n_estimators": [100, 300], "learning_rate": [0.05, 0.1]},
        -1
    ),
}




In [ ]:
import numpy as np
import pandas as pd

# Daten laden
copper = pd.read_csv("./data/copper_prices.csv")
copper["log_price"] = np.log(copper["price"])

# Datum generieren (monatlich ab 1960)
copper["Date"] = pd.date_range(start="1960-01", periods=len(copper), freq="MS")

# Features: Lags
for lag in [1, 2, 3, 6, 12]:
    copper[f"lag_{lag}"] = copper["log_price"].shift(lag)

copper = copper.dropna().reset_index(drop=True)

In [ ]:

import sys
# sys.path not needed after pip install qtml
from qtml.Time_Series_Univariate import rolling_forecast


# ══════════════════════════════════════════════════════════════════════
# 7. ROLLING FORECAST — Letzte 6 Monate als Test
# ══════════════════════════════════════════════════════════════════════


predictions = rolling_forecast(
    copper[["Date", "log_price"] + [f"lag_{l}" for l in [1,2,3,6,12]]],
    target="log_price",
    models=models,
    test_start="2006-12-01",
    test_end=copper["Date"].max().strftime("%Y-%m-%d"),
    cv=2
)




In [ ]:
# ??? idk if correct

import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import RidgeCV

def ensemble_forecast(predictions, y_test, 
                      methods=["voting", "weighted", "stacking", "adaptive"], 
                      warmup=60, models=None, val_ratio=0.5):
    """
    Ensemble-Methoden auf rolling_forecast() Predictions anwenden.
    
    Parameters
    ----------
    predictions : dict
        Output von rolling_forecast() → {model_name: [pred_t1, pred_t2, ...]}
    y_test : array
        Tatsächliche Werte im Testzeitraum
    methods : list
        "voting"   → Simple Average
        "weighted" → Adaptive Inverse-RMSE Gewichte (nutzt y_test[:t] für Gewichte)
        "stacking" → Echtes Stacking (Train/Val Split, y_test nur in Val-Hälfte)
        "adaptive" → Online Stacking (Meta-Learner updated bei jedem t, nutzt y_test[:t])
    warmup : int
        Ab wann Adaptive/Weighted starten (brauchen History)
    models : list, optional
        Welche Base Models nutzen. Default: alle.
    val_ratio : float
        Anteil der Testperiode für Meta-Learner Training bei echtem Stacking.
        z.B. 0.5 → erste 50% = Val, letzte 50% = echtes OOS
    """
    # Modelle auswählen
    if models is None:
        model_names = list(predictions.keys())
    else:
        model_names = [name for name in models if name in predictions]
    
    pred_matrix = np.column_stack([predictions[name] for name in model_names])
    T, n_models = pred_matrix.shape
    y_test = np.array(y_test)
    
    results = {}
    
    # ══════════════════════════════════════════════════════════════════════
    # 1. VOTING: Simple Average
    # ══════════════════════════════════════════════════════════════════════
    if "voting" in methods:
        voting_pred = pred_matrix.mean(axis=1)
        results["Voting (Average)"] = voting_pred
    
    # ══════════════════════════════════════════════════════════════════════
    # 2. WEIGHTED VOTING: Adaptive Inverse-RMSE Gewichte (Rolling)
    #    → Nutzt y_test[:t] für Gewichtberechnung
    # ══════════════════════════════════════════════════════════════════════
    if "weighted" in methods:
        weighted_pred = np.zeros(T)
        
        for t in range(T):
            if t < warmup:
                weighted_pred[t] = pred_matrix[t].mean()
            else:
                recent_preds = pred_matrix[t-warmup:t]
                recent_true  = y_test[t-warmup:t]
                
                rmses = np.array([
                    np.sqrt(mean_squared_error(recent_true, recent_preds[:, i]))
                    for i in range(n_models)
                ])
                rmses = np.clip(rmses, 1e-10, None)
                weights = 1.0 / rmses
                weights = weights / weights.sum()
                
                weighted_pred[t] = np.dot(pred_matrix[t], weights)
        
        results["Voting (Weighted)"] = weighted_pred
    
    # ══════════════════════════════════════════════════════════════════════
    # 3. STACKING (True): Train/Val Split — y_test NUR in Val-Hälfte
    #    → Echtes Stacking: Meta-Learner trainiert auf Val, predictet OOS
    # ══════════════════════════════════════════════════════════════════════
    if "stacking" in methods:
        stack_pred = np.zeros(T)
        meta = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0])
        
        val_end = int(T * val_ratio)
        
        # Meta-Learner auf erste Hälfte trainieren
        X_meta = pred_matrix[:val_end]
        y_meta = y_test[:val_end]
        meta.fit(X_meta, y_meta)
        
        # Alle Predictions mit fixem Meta-Learner
        stack_pred[:] = meta.predict(pred_matrix)
        
        results["Stacking (True)"] = stack_pred
    
    # ══════════════════════════════════════════════════════════════════════
    # 4. ADAPTIVE: Online Meta-Learner (Rolling Ridge)
    #    → Nutzt y_test[:t] — kein echtes Stacking, aber adaptiv
    # ══════════════════════════════════════════════════════════════════════
    if "adaptive" in methods:
        adapt_pred = np.zeros(T)
        meta_adapt = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0])
        
        for t in range(T):
            if t < warmup:
                adapt_pred[t] = pred_matrix[t].mean()
            else:
                meta_adapt.fit(pred_matrix[:t], y_test[:t])
                adapt_pred[t] = meta_adapt.predict(pred_matrix[t:t+1])[0]
        
        results["Adaptive (Online Ridge)"] = adapt_pred
    
    # ══════════════════════════════════════════════════════════════════════
    # EVALUATION
    # ══════════════════════════════════════════════════════════════════════
    print("=" * 80)
    print("ENSEMBLE RESULTS")
    print("=" * 80)
    
    # Base Models
    print("\n--- Base Models ---")
    for name in model_names:
        preds = np.array(predictions[name])
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2 = 1 - np.sum((y_test - preds)**2) / np.sum((y_test - y_test.mean())**2)
        print(f"{name:30s} RMSE={rmse:.4f}  R²={r2:.4f}")
    
    # Ensembles
    print("\n--- Ensemble Methods ---")
    for name, preds in results.items():
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2 = 1 - np.sum((y_test - preds)**2) / np.sum((y_test - y_test.mean())**2)
        print(f"{name:30s} RMSE={rmse:.4f}  R²={r2:.4f}")
    
    # Stacking Details
    if "stacking" in methods:
        print(f"\nStacking (True) Meta-Gewichte: {dict(zip(model_names, meta.coef_.round(4)))}")
        print(f"Stacking (True) Alpha: {meta.alpha_}")
        print(f"Stacking (True) Val-Periode: t=0 bis t={val_end} | OOS: t={val_end} bis t={T}")
    
    if "adaptive" in methods and hasattr(meta_adapt, 'coef_'):
        print(f"\nAdaptive Meta-Gewichte (letzte): {dict(zip(model_names, meta_adapt.coef_.round(4)))}")
        print(f"Adaptive Alpha: {meta_adapt.alpha_}")
    
    return results




# ══════════════════════════════════════════════════════════════════════
# 8. ENSEMBLE
# ══════════════════════════════════════════════════════════════════════
date_col = "Date"
y_test = ts[ts[date_col] >= "2025-06-01"]["log_return"].values

ensemble_results = ensemble_forecast(
    predictions, y_test,
    methods=["voting", "weighted", "stacking", "adaptive"],
    models=["XGBoost", "LightGBM", "Ridge"],
    warmup=60,
    val_ratio=0.5
)

